# AML QQ PLOTS 
 
Quantile–quantile plots and genomic inflation values for summary statistics from set-based tests using SAIGE-GENE+ are calculated for each maximum minor allele frequency threshold and test type, including burden, SKAT, and SKAT-O.
- Table 1 : Genomic inflation 
- Plot 1 : Quantile-quantile plot 

In [1]:
# Install packages 
if (!requireNamespace("flextable", quietly = TRUE)) {
  install.packages("flextable")
}

if (!requireNamespace("officer", quietly = TRUE)) {
  install.packages("officer")
}

if (!requireNamespace("janitor", quietly = TRUE)) {
  install.packages("janitor")
}

if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

BiocManager::install("biomaRt")

install.packages("xlsx")

install.packages("paletteer")

Bioconductor version '3.21' is out-of-date; the current release version '3.23'
  is available with R version '4.6'; see https://bioconductor.org/install

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cloud.r-project.org

Bioconductor version 3.21 (BiocManager 1.30.25), R 4.5.0 (2025-04-11)

Warning message:
“package(s) not installed when version(s) same as or greater than current; use
  `force = TRUE` to re-install: 'biomaRt'”
Old packages: 'alabaster.base', 'AnnotationHub', 'AnVIL', 'backports',
  'base64enc', 'BH', 'bigrquery', 'BiocFileCache', 'BiocGenerics',
  'BiocManager', 'BiocParallel', 'bit64', 'blob', 'boot', 'broom', 'bslib',
  'cli', 'clock', 'cluster', 'colorspace', 'commonmark', 'CompQuadForm',
  'cowplot', 'cpp11', 'credentials', 'crosstalk', 'curl', 'data.table', 'DBI',
  'dbplyr', 'DelayedArray', 'DESeq2', 'devtools', 'diffobj', 'digest',
  '

In [2]:
# Load the package
library(tidyverse)
library(ggplot2)
library(data.table)
library(purrr)
# Install these packages 
library(janitor)
library(flextable)
library(officer)
library(biomaRt)
library(patchwork)
library(xlsx)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘data.table’


The following objects are masked from ‘package:lubridate’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The following objects are masked from ‘package:dplyr’:

    between, first, last


The following object is masked from ‘package:purrr’:

    transpose



Attaching package: ‘janitor’


The following objects are masked from ‘package:stats’:

    chisq.test, fisher.test



Attaching package: ‘fle

In [3]:
# Function: Formats the gene set-based rare variant assoication test results 
process_results <- function(results) { 
    results %>% 
    filter(Group != "Cauchy") %>% 
    # inner_join(gene_symbols, by = "Region") %>% 
    mutate(max_MAF = as.character(max_MAF)) %>% 
    mutate(negative_log_p_values = -log10(Pvalue)) %>% 
    # mutate(Chromosome = gsub("chr", "", Chromosome)) %>% 
    mutate(Chromosome = as.numeric(Chromosome)) %>% 
    mutate(Region = as.factor(Region)) %>% 
    mutate(Region_factor = fct_reorder(Region, Chromosome))
  }

# HIGH CONFIDENCE ANNOATIONS 

In [4]:
# ========================================================================================================
# High confidence annoations (part 1)
# ========================================================================================================

# Load data from workspace bucket  
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

files <- c()
stats <- c() 

system("mkdir -p local_dir") 
phenotypes <- c("aml_phenotype",
                "aml_biosample_age", 
                "aml_diagnosis_age")

for (pheno in phenotypes){
    
    for (chr in 1:22){ 
        
        print(paste0("Loading test:", pheno))
        print(paste0("Loading chromosome:", chr))

        cmd <- paste0(
            "gsutil -m cp -r ",
            my_bucket,
            "/data/saige_step02/HC_analysis/EUR_", 
            pheno, 
            "_high_confidence_chr",
            chr,
            " local_dir"
        )

        system(cmd)

        path <- paste0("local_dir/EUR_",
                       pheno,
                       "_high_confidence_chr", chr)
        f <- fread(path)
        
        f$Chromosome <- chr
        
        processed <- process_results(f)

        files[[chr]] <- processed
    }
    
    stat <- do.call(rbind, files)
    stats[[paste0(pheno,"_high_confidence")]] <- stat
}


[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:1"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:2"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:3"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:4"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:5"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:6"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:7"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:8"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:9"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:10"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:11"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:12"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:13"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:14"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:15"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:16"
[1] "Loading test:aml_phenotype"


In [5]:
# ========================================================================================================
# High confidence annoations (part 1)
# ========================================================================================================

# Load data from workspace bucket  
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

system("mkdir -p local_dir") 
phenotypes <- c("biosample_age")

for (pheno in phenotypes){
    
    for (chr in 1:22){ 
        
        print(paste0("Loading test: ", pheno))
        print(paste0("Loading chromosome: ", chr))

        cmd <- paste0(
            "gsutil -m cp -r ",
            my_bucket,
            "/data/saige_step02/biosample_collection_check/EUR_",
            pheno, 
            "_high_confidence_chr",
            chr,
            " local_dir"
        )

        system(cmd)

        path <- paste0("local_dir/EUR_",
                       pheno,
                       "_high_confidence_chr", chr)
        f <- fread(path)
        
        f$Chromosome <- chr
        
        processed <- process_results(f)

        files[[chr]] <- processed
    }
    
    stat <- do.call(rbind, files)
    stats[[paste0(pheno,"_high_confidence")]] <- stat
}


[1] "Loading test: biosample_age"
[1] "Loading chromosome: 1"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 2"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 3"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 4"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 5"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 6"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 7"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 8"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 9"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 10"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 11"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 12"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 13"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 14"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 15"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 16"
[

In [6]:
# ========================================================================================================
# High confidence annoations (part 2)
# ========================================================================================================

# Load data from workspace bucket  
system("mkdir -p local_dir") 
covariates <- c("_deprivation_index",
                "_deprivation")

for (cov in covariates){
    
    for (chr in 1:22){ 
        
        print(paste0("Loading test:", cov))
        print(paste0("Loading chromosome:", chr))

        cmd <- paste0(
            "gsutil -m cp -r ",
            my_bucket,
            "/data/saige_step02/HC_analysis/EUR_", 
            "aml_phenotype", 
            "_high_confidence_chr",
            chr, 
            cov, 
            " local_dir"
        )

        system(cmd)

        path <- paste0("local_dir/EUR_",
                       "aml_phenotype",
                       "_high_confidence_chr", chr, cov)
        f <- fread(path)
        
        f$Chromosome <- chr
        
        processed <- process_results(f)

        files[[chr]] <- processed
    }
    
    stat <- do.call(rbind, files)
    stats[[paste0(cov,"_high_confidence")]] <- stat
}


[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:1"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:2"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:3"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:4"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:5"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:6"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:7"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:8"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:9"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:10"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:11"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:12"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:13"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:14"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:15"
[1] "Loading test:_

# SYNONYMOUS ANNOATIONS

In [7]:
# ========================================================================================================
# Synonymous annoations (part 1)
# ========================================================================================================

# Load data from workspace bucket  
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

system("mkdir -p local_dir") 
phenotypes <- c("aml_phenotype",
                "aml_biosample_age", 
                "aml_diagnosis_age")

for (pheno in phenotypes){
    
    for (chr in 1:22){ 
        
        print(paste0("Loading test:", pheno))
        print(paste0("Loading chromosome:", chr))

        cmd <- paste0(
            "gsutil -m cp -r ",
            my_bucket,
            "/data/saige_step02/HC_analysis/EUR_", 
            pheno, 
            "_synonymous_chr",
            chr,
            " local_dir"
        )

        system(cmd)

        path <- paste0("local_dir/EUR_",
                       pheno,
                       "_synonymous_chr", chr)
        f <- fread(path)
        
        f$Chromosome <- chr
        
        processed <- process_results(f)

        files[[chr]] <- processed
    }
    
    stat <- do.call(rbind, files)
    stats[[paste0(pheno,"_synonymous")]] <- stat
}


[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:1"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:2"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:3"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:4"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:5"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:6"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:7"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:8"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:9"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:10"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:11"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:12"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:13"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:14"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:15"
[1] "Loading test:aml_phenotype"
[1] "Loading chromosome:16"
[1] "Loading test:aml_phenotype"


In [8]:
# ========================================================================================================
# Synonymous annoations (part 1)
# ========================================================================================================

# Load data from workspace bucket  
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

system("mkdir -p local_dir") 

phenotypes <- c("biosample_age")

for (pheno in phenotypes){
    
    files <- list()   # reset per phenotype
    
    for (chr in 1:22){ 
        
        print(paste0("Loading test: ", pheno))
        print(paste0("Loading chromosome: ", chr))

        cmd <- paste0(
            "gsutil -m cp -r ",
            my_bucket,
            "/data/saige_step02/biosample_collection_check/EUR_",
            pheno, 
            "_synonymous_chr",
            chr,
            " local_dir/"
        )

        system(cmd)

        path <- paste0(
            "local_dir/EUR_",
            pheno,
            "_synonymous_chr", chr
        )

        # If multiple files exist (common with SAIGE output)
        if (dir.exists(path)) {
            file_list <- list.files(path, full.names = TRUE)
            f <- rbindlist(lapply(file_list, fread), fill = TRUE)
        } else {
            f <- fread(path)
        }
        
        f$Chromosome <- chr
        
        processed <- process_results(f)

        files[[chr]] <- processed
    }
    
    stat <- rbindlist(files, fill = TRUE)
    stats[[paste0(pheno, "_synonymous")]] <- stat
}



[1] "Loading test: biosample_age"
[1] "Loading chromosome: 1"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 2"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 3"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 4"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 5"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 6"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 7"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 8"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 9"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 10"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 11"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 12"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 13"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 14"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 15"
[1] "Loading test: biosample_age"
[1] "Loading chromosome: 16"
[

In [9]:
# ========================================================================================================
# Synonymous annoations (part 2)
# ========================================================================================================

# Load data from workspace bucket  
system("mkdir -p local_dir") 
covariates <- c("_deprivation_index",
                "_deprivation")

for (cov in covariates){
    
    for (chr in 1:22){ 
        
        print(paste0("Loading test:", cov))
        print(paste0("Loading chromosome:", chr))

        cmd <- paste0(
            "gsutil -m cp -r ",
            my_bucket,
            "/data/saige_step02/HC_analysis/EUR_", 
            "aml_phenotype", 
            "_synonymous_chr",
            chr, 
            cov, 
            " local_dir"
        )

        system(cmd)

        path <- paste0("local_dir/EUR_",
                       "aml_phenotype",
                       "_synonymous_chr", chr, cov)
        f <- fread(path)
        
        f$Chromosome <- chr
        
        processed <- process_results(f)

        files[[chr]] <- processed
    }
    
    stat <- do.call(rbind, files)
    stats[[paste0(cov,"_synonymous")]] <- stat
}


[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:1"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:2"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:3"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:4"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:5"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:6"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:7"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:8"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:9"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:10"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:11"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:12"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:13"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:14"
[1] "Loading test:_deprivation_index"
[1] "Loading chromosome:15"
[1] "Loading test:_

# GENOMIC INFLATION 

In [10]:
# Clean name of list 
clean_names <- function(x) {
  x <- gsub("_", " ", x)
  
  sapply(strsplit(x, " "), function(w) {
    paste(toupper(substring(w, 1, 1)), substring(w, 2), sep = "", collapse = " ")
  })
}

new_names <- c(
  "aml_phenotype_high_confidence", 
  "aml_biosample_age_high_confidence", 
  "aml_diagnosis_age_high_confidence", 
  "biosample_age_components_high_confidence", 
  "aml_deprivation_index_high_confidence",
  "aml_deprivation_components_high_confidence",
  
  "aml_phenotype_synonymous", 
  "aml_biosample_age_synonymous", 
  "aml_diagnosis_age_synonymous", 
  "biosample_age_components_synonymous", 
  "aml_deprivation_index_synonymous",
  "aml_deprivation_components_synonymous"
)

names(stats) <- clean_names(new_names)

In [11]:
# ========================================================================================================
# Table 1 : Genomic inflation 
# ========================================================================================================

df_list <- stats

genomic_inflation_list <- list()

for (df_name in names(df_list)) {
    
    df <- df_list[[df_name]]  # get the current data frame
    # Transform the full dataset for plotting
    genomic_inflation <- 
    df %>%
    filter(!is.na(Pvalue) & Pvalue > 0 & Pvalue <= 1) %>%
    pivot_longer(
    cols = starts_with("Pvalue"),
    names_to = "Statistical_Test",
    names_prefix = "Pvalue_",
    values_to = "P_value",
    values_drop_na = TRUE
    ) %>%
     mutate(
         Statistical_Test = ifelse(Statistical_Test == "Pvalue", "SKAT-O", Statistical_Test)
     ) %>%
    mutate(max_MAF = case_when(
    max_MAF == "0.01"   ~ "1 x 10⁻²",
    max_MAF == "0.001"  ~ "1 x 10⁻³",
    max_MAF == "1e-04"  ~ "1 x 10⁻⁴",
    TRUE ~ max_MAF)) %>%
    group_by(max_MAF, Statistical_Test) %>% 
    mutate(chisq = qchisq(P_value,1,lower.tail=FALSE)) %>% 
    group_by(max_MAF, Statistical_Test) %>% 
    mutate(lambda = (median(chisq) / qchisq(0.5,1)),
           lambda_0.01 = quantile(chisq, probs = 0.99) / qchisq(0.99, df = 1)) %>% 
    distinct(Statistical_Test,max_MAF,lambda,lambda_0.01) %>% 
    rename("Statistical Test" = "Statistical_Test", 
         "Maximum MAF" = "max_MAF", 
         "Lambda" = "lambda",
         "Lambda 0.01" = "lambda_0.01") 

    genomic_inflation$Analysis <- df_name
    genomic_inflation_list[[df_name]] <- genomic_inflation

}


In [12]:
# Combine genomic inflation data frames 
genomic_inflation_df <- do.call(rbind, genomic_inflation_list)

genomic_inflation_df <- genomic_inflation_df[, c(
  "Analysis", 
  "Statistical Test", 
  "Maximum MAF", 
  "Lambda", 
  "Lambda 0.01"
)]

# Create column description sheet
column_descriptions <- data.frame(
  Column = c(
    "Analysis",
    "Statistical Test",
    "Maximum MAF",
    "Lambda",
    "Lambda 0.01"
  ),
  Description = c(
    "Name of the analysis performed",
    "Statistical test used in the analysis",
    "Maximum minor allele frequency threshold used for variant inclusion",
    "Genomic inflation factor (lambda GC)",
    "Genomic inflation factor for the top 1% of p-values"
  )
)

# Write Excel with multiple sheets
library(openxlsx)

wb <- createWorkbook()

# First sheet
addWorksheet(wb, "Genomic Inflation")
writeData(wb, "Genomic Inflation", genomic_inflation_df)

# Create the second sheet BEFORE writing to it
addWorksheet(wb, "Column Descriptions")

writeData(
  wb,
  "Column Descriptions",
  column_descriptions
)

saveWorkbook(
  wb,
  "genomic_inflation.xlsx",
  overwrite = TRUE
)


Attaching package: ‘openxlsx’


The following objects are masked from ‘package:xlsx’:

    createWorkbook, loadWorkbook, read.xlsx, saveWorkbook, write.xlsx


